In [1]:
!pip install datasets wandb regex -q

In [2]:
import os, math, time, json, random, re
import regex
from collections import defaultdict
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import wandb
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Using device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


### BPE Tokenizer (from scratch)

In [3]:
class BPETokenizer:
    """
    Byte-Pair Encoding tokenizer built entirely from scratch.
    Follows the GPT-2 approach: byte-level BPE with a regex pre-tokenizer.
    """

    GPT2_PATTERN = regex.compile(
        r"""'s|'t|'re|'ve|'m|'ll|'d| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""
    )

    def __init__(self, vocab_size: int = 4096):
        self.vocab_size = vocab_size
        self.merges = {}
        self.vocab = {}
        self.inv_vocab = {}
        self._build_byte_vocab()

    def _build_byte_vocab(self):
        for i in range(256):
            self.vocab[i] = bytes([i])
            self.inv_vocab[bytes([i])] = i

    def _text_to_ids(self, text: str):
        return list(text.encode('utf-8', errors='replace'))

    def _pre_tokenize(self, text: str):
        return regex.findall(self.GPT2_PATTERN, text)

    def _get_stats(self, corpus):
        counts = defaultdict(int)
        for word_ids in corpus:
            for a, b in zip(word_ids, word_ids[1:]):
                counts[(a, b)] += 1
        return counts

    def _merge_corpus(self, corpus, pair, new_id):
        a, b = pair
        new_corpus = []
        for word_ids in corpus:
            merged = []
            i = 0
            while i < len(word_ids):
                if i < len(word_ids) - 1 and word_ids[i] == a and word_ids[i+1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(word_ids[i])
                    i += 1
            new_corpus.append(merged)
        return new_corpus

    def train(self, texts: list, verbose: bool = True):
        print(f'Building corpus from {len(texts):,} texts...')
        corpus = []
        for text in texts:
            for chunk in self._pre_tokenize(text):
                corpus.append(self._text_to_ids(chunk))

        print(f'Corpus: {sum(len(w) for w in corpus):,} tokens | Starting merges...')
        num_merges = self.vocab_size - 256

        for step in range(num_merges):
            stats = self._get_stats(corpus)
            if not stats:
                break
            best_pair = max(stats, key=stats.get)
            new_id = 256 + step
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.inv_vocab[self.vocab[new_id]] = new_id
            corpus = self._merge_corpus(corpus, best_pair, new_id)

            if verbose and (step + 1) % 200 == 0:
                token_str = self.vocab[new_id][:20]
                print(f'  Merge {step+1}/{num_merges}: {best_pair} -> {new_id} '
                      f'("{token_str}") freq={stats[best_pair]:,}')

        print(f'Done! Vocabulary size: {len(self.vocab):,}')

    def encode(self, text: str) -> list:
        ids = []
        for chunk in self._pre_tokenize(text):
            chunk_ids = self._text_to_ids(chunk)
            changed = True
            while changed and len(chunk_ids) > 1:
                changed = False
                new_ids = []
                i = 0
                while i < len(chunk_ids):
                    if i < len(chunk_ids) - 1:
                        pair = (chunk_ids[i], chunk_ids[i+1])
                        if pair in self.merges:
                            new_ids.append(self.merges[pair])
                            i += 2
                            changed = True
                            continue
                    new_ids.append(chunk_ids[i])
                    i += 1
                chunk_ids = new_ids
            ids.extend(chunk_ids)
        return ids

    def decode(self, ids: list) -> str:
        raw = b''.join(self.vocab.get(i, b'?') for i in ids)
        return raw.decode('utf-8', errors='replace')

    def save(self, path: str):
        data = {
            'vocab_size': self.vocab_size,
            'merges': {f'{a},{b}': c for (a, b), c in self.merges.items()},
            'vocab': {str(k): list(v) for k, v in self.vocab.items()}
        }
        with open(path, 'w') as f:
            json.dump(data, f)
        print(f'Tokenizer saved to {path}')

    @classmethod
    def load(cls, path: str):
        with open(path) as f:
            data = json.load(f)
        tok = cls(vocab_size=data['vocab_size'])
        tok.merges = {tuple(int(x) for x in k.split(',')): v
                      for k, v in data['merges'].items()}
        tok.vocab = {int(k): bytes(v) for k, v in data['vocab'].items()}
        tok.inv_vocab = {v: k for k, v in tok.vocab.items()}
        return tok

print('BPETokenizer defined.')

BPETokenizer defined.


### Load Python dataset

In [4]:
print('Loading Python code dataset...')
ds = load_dataset('code_search_net', 'python', split='train', trust_remote_code=True)
print(f'Dataset size: {len(ds):,} examples')

all_texts = []
for ex in ds:
    code = ex.get('whole_func_string', '') or ex.get('func_code_string', '')
    if code and len(code) > 50:
        all_texts.append(code)

print(f'Collected {len(all_texts):,} Python functions')
print(f'Total characters: {sum(len(t) for t in all_texts):,}')
print('\nSample:')
print(all_texts[0][:400])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading Python code dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

...

python/validation-00000-of-00001.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

Dataset size: 412,178 examples
Collected 412,178 Python functions
Total characters: 435,436,592

Sample:
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level i


### Train BPE tokenizer

In [6]:
tokenizer_texts = all_texts[:3000]

tokenizer = BPETokenizer(vocab_size=2048)
tokenizer.train(tokenizer_texts, verbose=True)
tokenizer.save('/content/tokenizer.json')

# Sanity check
test = 'def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)'
encoded = tokenizer.encode(test)
decoded = tokenizer.decode(encoded)
print(f'\nEncode/decode test:')
print(f'  Original    : {repr(test[:60])}')
print(f'  Tokens      : {encoded[:20]}...')
print(f'  Decoded     : {repr(decoded[:60])}')
print(f'  Match       : {test == decoded}')
print(f'  Compression : {len(test)} chars -> {len(encoded)} tokens ({len(encoded)/len(test):.2f} tok/char)')

Building corpus from 3,000 texts...
Corpus: 2,801,182 tokens | Starting merges...
  Merge 200/1792: (116, 315) -> 455 ("b'tent'") freq=1,458
  Merge 400/1792: (46, 46) -> 655 ("b'..'") freq=648
  Merge 600/1792: (32, 87) -> 855 ("b' W'") freq=414
  Merge 800/1792: (280, 893) -> 1055 ("b' current'") freq=283
  Merge 1000/1792: (668, 115) -> 1255 ("b'tings'") freq=208
  Merge 1200/1792: (322, 303) -> 1455 ("b' but'") freq=164
  Merge 1400/1792: (1212, 395) -> 1655 ("b' Args'") freq=132
  Merge 1600/1792: (290, 613) -> 1855 ("b'itive'") freq=110
Done! Vocabulary size: 2,048
Tokenizer saved to /content/tokenizer.json

Encode/decode test:
  Original    : 'def fibonacci(n):\n    if n <= 1:\n        return n\n    return'
  Tokens      : [337, 283, 947, 264, 1112, 692, 40, 110, 311, 271, 307, 293, 851, 61, 473, 58, 263, 543, 536, 319]...
  Decoded     : 'def fibonacci(n):\n    if n <= 1:\n        return n\n    return'
  Match       : True
  Compression : 92 chars -> 46 tokens (0.50 tok/char)


### Model config

In [7]:
@dataclass
class ModelConfig:
    vocab_size: int = 2048      # changed from 4096
    context_len: int = 256
    n_layers: int = 6
    n_heads: int = 6
    d_model: int = 384
    d_ff: int = 1536
    dropout: float = 0.1
    bias: bool = False

    @property
    def d_head(self):
        return self.d_model // self.n_heads

    def num_params(self):
        embed = self.vocab_size * self.d_model
        attn  = self.n_layers * (4 * self.d_model * self.d_model)
        ffn   = self.n_layers * (3 * self.d_model * self.d_ff)
        norm  = self.n_layers * 2 * self.d_model
        return embed + attn + ffn + norm

cfg = ModelConfig()
print(f'Layers     : {cfg.n_layers}')
print(f'Heads      : {cfg.n_heads}')
print(f'd_model    : {cfg.d_model}')
print(f'd_head     : {cfg.d_head}')
print(f'Context len: {cfg.context_len}')
print(f'~Params    : {cfg.num_params()/1e6:.1f}M')

Layers     : 6
Heads      : 6
d_model    : 384
d_head     : 64
Context len: 256
~Params    : 14.9M


### Rotary Positional Embeddings


In [8]:
class RotaryEmbedding(nn.Module):
    """
    RoPE: rotate Q and K by position-dependent angles.
    Encodes relative position into attention — better than learned absolute embeddings.
    """
    def __init__(self, d_head: int, max_len: int = 2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer('inv_freq', inv_freq)
        self._build_cache(max_len, d_head)

    def _build_cache(self, max_len, d_head):
        t = torch.arange(max_len).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cache', emb.cos())
        self.register_buffer('sin_cache', emb.sin())

    def _rotate_half(self, x):
        half = x.shape[-1] // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len: int):
        cos = self.cos_cache[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cache[:seq_len].unsqueeze(0).unsqueeze(0)
        return x * cos + self._rotate_half(x) * sin

print('RotaryEmbedding defined.')

RotaryEmbedding defined.


### Causal Self-Attention

In [9]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0

        self.n_heads = cfg.n_heads
        self.d_head  = cfg.d_head
        self.d_model = cfg.d_model
        self.dropout = cfg.dropout

        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.out_proj  = nn.Linear(cfg.d_model, cfg.d_model,     bias=cfg.bias)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.rope      = RotaryEmbedding(cfg.d_head, max_len=cfg.context_len)

        mask = torch.tril(torch.ones(cfg.context_len, cfg.context_len))
        self.register_buffer('causal_mask', mask.view(1, 1, cfg.context_len, cfg.context_len))

        self.cache_k = None
        self.cache_v = None

    def forward(self, x, use_cache=False):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.d_model, dim=-1)

        def reshape(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = reshape(q), reshape(k), reshape(v)

        q = self.rope(q, T)
        k = self.rope(k, T)

        if use_cache:
            if self.cache_k is None:
                self.cache_k, self.cache_v = k, v
            else:
                self.cache_k = torch.cat([self.cache_k, k], dim=2)
                self.cache_v = torch.cat([self.cache_v, v], dim=2)
            k, v = self.cache_k, self.cache_v

        T_k = k.shape[2]
        scale = 1.0 / math.sqrt(self.d_head)
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.masked_fill(self.causal_mask[:, :, :T, :T_k] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

    def clear_cache(self):
        self.cache_k = self.cache_v = None

print('CausalSelfAttention defined.')

CausalSelfAttention defined.


### SwiGLU FFN + TransformerBlock + Full Model

In [10]:
class SwiGLU(nn.Module):
    """FFN used in LLaMA/PaLM — outperforms vanilla ReLU."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj   = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model,  bias=cfg.bias)
        self.drop      = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class TransformerBlock(nn.Module):
    """Pre-norm block: x = x + Attn(Norm(x)), x = x + FFN(Norm(x))"""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)

    def forward(self, x, use_cache=False):
        x = x + self.attn(self.norm1(x), use_cache=use_cache)
        x = x + self.ffn(self.norm2(x))
        return x


class MiniLLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.embed   = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm    = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying — share embedding and output weights (standard trick)
        self.lm_head.weight = self.embed.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape
        x = self.drop(self.embed(idx))
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size), targets.view(-1))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def clear_kv_cache(self):
        for block in self.blocks:
            block.attn.clear_cache()

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8, top_k=50):
        self.eval()
        self.clear_kv_cache()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        self.clear_kv_cache()
        return idx


model = MiniLLM(cfg).to(device)
print(f'Model built! Parameters: {model.num_params()/1e6:.2f}M')

Model built! Parameters: 14.95M


### Dataset

In [11]:
class CodeDataset(Dataset):
    """
    Tokenizes corpus into one flat array, yields fixed-length chunks.
    Packed format — zero padding waste.
    """
    def __init__(self, texts, tokenizer, context_len, max_texts=None):
        if max_texts:
            texts = texts[:max_texts]
        print(f'Tokenizing {len(texts):,} texts...')
        all_tokens = []
        for i, text in enumerate(texts):
            tokens = tokenizer.encode(text)
            all_tokens.extend(tokens)
            all_tokens.append(0)
            if (i + 1) % 5000 == 0:
                print(f'  {i+1:,}/{len(texts):,} | {len(all_tokens):,} tokens')
        self.data = torch.tensor(all_tokens, dtype=torch.long)
        self.context_len = context_len
        print(f'Dataset ready: {len(self.data):,} tokens -> {len(self):,} chunks')

    def __len__(self):
        return (len(self.data) - 1) // self.context_len

    def __getitem__(self, idx):
        start = idx * self.context_len
        x = self.data[start : start + self.context_len]
        y = self.data[start + 1 : start + self.context_len + 1]
        return x, y


random.shuffle(all_texts)
n_val = int(0.05 * len(all_texts))
val_texts   = all_texts[:n_val]
train_texts = all_texts[n_val:]

train_ds = CodeDataset(train_texts, tokenizer, cfg.context_len, max_texts=30000)
val_ds   = CodeDataset(val_texts,   tokenizer, cfg.context_len, max_texts=2000)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,}')

Tokenizing 30,000 texts...
  5,000/30,000 | 2,099,761 tokens
  10,000/30,000 | 4,264,144 tokens
  15,000/30,000 | 6,356,103 tokens
  20,000/30,000 | 8,486,235 tokens
  25,000/30,000 | 10,635,222 tokens
  30,000/30,000 | 12,813,272 tokens
Dataset ready: 12,813,272 tokens -> 50,051 chunks
Tokenizing 2,000 texts...
Dataset ready: 842,826 tokens -> 3,292 chunks
Train batches: 1,565 | Val batches: 103


### Trainer

In [12]:
class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg):
        self.model        = model
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.cfg          = cfg
        self.max_epochs   = 3
        self.lr           = 3e-4
        self.min_lr       = 3e-5
        self.weight_decay = 0.1
        self.grad_clip    = 1.0
        self.eval_every   = 200
        self.log_every    = 50
        self.save_path    = '/content/minillm_best.pt'
        self.step         = 0
        self.best_val     = float('inf')
        self.total_steps  = self.max_epochs * len(train_loader)

        decay_params    = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
        no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]
        self.optimizer = torch.optim.AdamW([
            {'params': decay_params,    'weight_decay': self.weight_decay},
            {'params': no_decay_params, 'weight_decay': 0.0}
        ], lr=self.lr, betas=(0.9, 0.95))

        self.scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

    def cosine_lr(self):
        warmup = int(0.05 * self.total_steps)
        if self.step < warmup:
            return self.lr * self.step / warmup
        progress = (self.step - warmup) / (self.total_steps - warmup)
        return self.min_lr + 0.5 * (self.lr - self.min_lr) * (1 + math.cos(math.pi * progress))

    def set_lr(self, lr):
        for g in self.optimizer.param_groups:
            g['lr'] = lr

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        total_loss = 0
        n = min(50, len(self.val_loader))
        for i, (x, y) in enumerate(self.val_loader):
            if i >= n: break
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                _, loss = self.model(x, y)
            total_loss += loss.item()
        self.model.train()
        return total_loss / n

    def save(self, val_loss):
        torch.save({'step': self.step, 'model': self.model.state_dict(),
                    'val_loss': val_loss, 'config': self.cfg}, self.save_path)
        print(f'  Checkpoint saved — val_loss={val_loss:.4f}')

    def train(self):
        print(f'Training {self.model.num_params()/1e6:.1f}M params for {self.max_epochs} epochs...')
        self.model.train()
        t0 = time.time()

        for epoch in range(self.max_epochs):
            for x, y in self.train_loader:
                x, y = x.to(device), y.to(device)
                lr = self.cosine_lr()
                self.set_lr(lr)

                self.optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                    _, loss = self.model(x, y)

                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.step += 1

                if self.step % self.log_every == 0:
                    elapsed = time.time() - t0
                    tok_s = self.log_every * self.train_loader.batch_size * self.cfg.context_len / elapsed
                    ppl = math.exp(min(loss.item(), 20))
                    print(f'Ep {epoch+1} | Step {self.step:5d} | loss={loss.item():.4f} | '
                          f'ppl={ppl:.1f} | lr={lr:.2e} | {tok_s/1e3:.1f}k tok/s')
                    t0 = time.time()

                if self.step % self.eval_every == 0:
                    val_loss = self.evaluate()
                    print(f'  >> VAL loss={val_loss:.4f} | ppl={math.exp(min(val_loss,20)):.1f}')
                    if val_loss < self.best_val:
                        self.best_val = val_loss
                        self.save(val_loss)

        print(f'\nDone! Best val loss={self.best_val:.4f} | ppl={math.exp(self.best_val):.1f}')

### Train

In [13]:
trainer = Trainer(model, train_loader, val_loader, cfg)
trainer.train()

Training 15.0M params for 3 epochs...


/tmp/ipykernel_7029/1961182632.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
/tmp/ipykernel_7029/1961182632.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == 'cuda')):


Ep 1 | Step    50 | loss=6.7611 | ppl=863.6 | lr=6.28e-05 | 55.9k tok/s
Ep 1 | Step   100 | loss=5.8399 | ppl=343.7 | lr=1.27e-04 | 69.4k tok/s
Ep 1 | Step   150 | loss=4.9850 | ppl=146.2 | lr=1.91e-04 | 68.8k tok/s
Ep 1 | Step   200 | loss=4.3685 | ppl=78.9 | lr=2.55e-04 | 68.4k tok/s


/tmp/ipykernel_7029/1961182632.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == 'cuda')):


  >> VAL loss=4.3903 | ppl=80.7
  Checkpoint saved — val_loss=4.3903
Ep 1 | Step   250 | loss=4.0124 | ppl=55.3 | lr=3.00e-04 | 48.9k tok/s
Ep 1 | Step   300 | loss=3.6992 | ppl=40.4 | lr=3.00e-04 | 67.4k tok/s
Ep 1 | Step   350 | loss=3.4017 | ppl=30.0 | lr=3.00e-04 | 67.2k tok/s
Ep 1 | Step   400 | loss=3.4902 | ppl=32.8 | lr=2.99e-04 | 66.9k tok/s
  >> VAL loss=3.3116 | ppl=27.4
  Checkpoint saved — val_loss=3.3116
Ep 1 | Step   450 | loss=3.2599 | ppl=26.0 | lr=2.98e-04 | 47.6k tok/s
Ep 1 | Step   500 | loss=3.2291 | ppl=25.3 | lr=2.98e-04 | 65.9k tok/s
Ep 1 | Step   550 | loss=2.9742 | ppl=19.6 | lr=2.97e-04 | 65.1k tok/s
Ep 1 | Step   600 | loss=2.9669 | ppl=19.4 | lr=2.96e-04 | 64.9k tok/s
  >> VAL loss=2.9547 | ppl=19.2
  Checkpoint saved — val_loss=2.9547
Ep 1 | Step   650 | loss=2.9781 | ppl=19.7 | lr=2.94e-04 | 45.5k tok/s
Ep 1 | Step   700 | loss=2.8850 | ppl=17.9 | lr=2.93e-04 | 63.6k tok/s
Ep 1 | Step   750 | loss=2.8138 | ppl=16.7 | lr=2.91e-04 | 63.2k tok/s
Ep 1 | Step 

### Generate Python code

In [14]:
def generate_code(prompt, max_tokens=250, temperature=0.8, top_k=50):
    model.eval()
    ids = tokenizer.encode(prompt)
    x   = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(x, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0].tolist())


prompts = [
    'def binary_search(arr, target):',
    'class Stack:\n    def __init__(self):',
    'def merge_sort(arr):\n    """Sort array using merge sort."""',
    'import numpy as np\n\ndef normalize(',
]

for prompt in prompts:
    print('=' * 60)
    print(f'PROMPT: {prompt}')
    print('-' * 60)
    print(generate_code(prompt))
    print()

PROMPT: def binary_search(arr, target):
------------------------------------------------------------
def binarysearch(arr, target):
    """
    Add a list of variants to the given parameters.

    :param parameters: Optional parameters to return.
    :param encoding:
    :return:
    """
    if encoding is None:
        encoding = 'NAME'
    elif encoding in ['NAME', 'NAME', 'None', 'NAME', 'NAME']:
        encoding = 'NAME'
    else:
        raise ValueError('Remote parameters not considered as named string')
    if encoding in ['NAME', 'NAME', 'NAME', 'NAME', 'NAME', 'NAME']:
        return encoding
    if encoding is True:
        return encoding
    else:
        return encoding
    # C

PROMPT: class Stack:
    def __init__(self):
------------------------------------------------------------
class Stack:
    def init(self):
        """
            Process the worker in the Worker's API.

        :type  self: `Client`
        """
        if self.parent is not None:
            retur

### Final perplexity

In [15]:
@torch.no_grad()
def compute_perplexity(loader, n_batches=100):
    model.eval()
    total, count = 0, 0
    for i, (x, y) in enumerate(loader):
        if i >= n_batches: break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total += loss.item()
        count += 1
    avg = total / count
    return avg, math.exp(avg)

train_loss, train_ppl = compute_perplexity(train_loader)
val_loss,   val_ppl   = compute_perplexity(val_loader)

print(f'Train loss={train_loss:.4f} | ppl={train_ppl:.2f}')
print(f'Val   loss={val_loss:.4f}   | ppl={val_ppl:.2f}')
print()
print('Perplexity guide:  <10 excellent | 10-30 solid | 30-100 early stage')

Train loss=1.7501 | ppl=5.76
Val   loss=1.8856   | ppl=6.59

Perplexity guide:  <10 excellent | 10-30 solid | 30-100 early stage


### Save to Google Drive

In [17]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/MiniLLM'
os.makedirs(save_dir, exist_ok=True)

torch.save({'model': model.state_dict(), 'config': cfg,
            'val_loss': val_loss, 'val_ppl': val_ppl},
           f'{save_dir}/model_final.pt')

tokenizer.save(f'{save_dir}/tokenizer.json')

summary = {
    'params_M': round(model.num_params()/1e6, 2),
    'vocab_size': cfg.vocab_size, 'n_layers': cfg.n_layers,
    'n_heads': cfg.n_heads, 'd_model': cfg.d_model,
    'val_loss': round(val_loss, 4), 'val_ppl': round(val_ppl, 2),
    'dataset': 'CodeSearchNet Python'
}
with open(f'{save_dir}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved to Google Drive!')
print(json.dumps(summary, indent=2))

Mounted at /content/drive
Tokenizer saved to /content/drive/MyDrive/MiniLLM/tokenizer.json
Saved to Google Drive!
{
  "params_M": 14.95,
  "vocab_size": 2048,
  "n_layers": 6,
  "n_heads": 6,
  "d_model": 384,
  "val_loss": 1.8856,
  "val_ppl": 6.59,
  "dataset": "CodeSearchNet Python"
}


### Add W&B Dashboard

In [18]:
# Cell 12 - Setup W&B then Train
import wandb

wandb.login()  # paste your API key when prompted

wandb.init(
    project='mini-llm-python',
    config={
        'vocab_size' : cfg.vocab_size,
        'n_layers'   : cfg.n_layers,
        'n_heads'    : cfg.n_heads,
        'd_model'    : cfg.d_model,
        'context_len': cfg.context_len,
        'params_M'   : round(model.num_params() / 1e6, 2),
        'dataset'    : 'CodeSearchNet Python',
    }
)

class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg):
        self.model        = model
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.cfg          = cfg
        self.max_epochs   = 3
        self.lr           = 3e-4
        self.min_lr       = 3e-5
        self.weight_decay = 0.1
        self.grad_clip    = 1.0
        self.eval_every   = 200
        self.log_every    = 50
        self.save_path    = '/content/minillm_best.pt'
        self.step         = 0
        self.best_val     = float('inf')
        self.total_steps  = self.max_epochs * len(train_loader)

        decay_params    = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
        no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]
        self.optimizer = torch.optim.AdamW([
            {'params': decay_params,    'weight_decay': self.weight_decay},
            {'params': no_decay_params, 'weight_decay': 0.0}
        ], lr=self.lr, betas=(0.9, 0.95))

        self.scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

    def cosine_lr(self):
        warmup = int(0.05 * self.total_steps)
        if self.step < warmup:
            return self.lr * self.step / warmup
        progress = (self.step - warmup) / (self.total_steps - warmup)
        return self.min_lr + 0.5 * (self.lr - self.min_lr) * (1 + math.cos(math.pi * progress))

    def set_lr(self, lr):
        for g in self.optimizer.param_groups:
            g['lr'] = lr

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        total_loss = 0
        n = min(50, len(self.val_loader))
        for i, (x, y) in enumerate(self.val_loader):
            if i >= n: break
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                _, loss = self.model(x, y)
            total_loss += loss.item()
        self.model.train()
        return total_loss / n

    def save(self, val_loss):
        torch.save({'step': self.step, 'model': self.model.state_dict(),
                    'val_loss': val_loss, 'config': self.cfg}, self.save_path)
        print(f'  Checkpoint saved — val_loss={val_loss:.4f}')

    def train(self):
        print(f'Training {self.model.num_params()/1e6:.1f}M params for {self.max_epochs} epochs...')
        self.model.train()
        t0 = time.time()

        for epoch in range(self.max_epochs):
            for x, y in self.train_loader:
                x, y = x.to(device), y.to(device)
                lr = self.cosine_lr()
                self.set_lr(lr)

                self.optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                    _, loss = self.model(x, y)

                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.step += 1

                if self.step % self.log_every == 0:
                    elapsed = time.time() - t0
                    tok_s = self.log_every * self.train_loader.batch_size * self.cfg.context_len / elapsed
                    ppl = math.exp(min(loss.item(), 20))
                    print(f'Ep {epoch+1} | Step {self.step:5d} | loss={loss.item():.4f} | '
                          f'ppl={ppl:.1f} | lr={lr:.2e} | {tok_s/1e3:.1f}k tok/s')
                    wandb.log({
                        'train/loss'     : loss.item(),
                        'train/ppl'      : ppl,
                        'train/lr'       : lr,
                        'train/tok_per_s': tok_s,
                        'step'           : self.step,
                    })
                    t0 = time.time()

                if self.step % self.eval_every == 0:
                    val_loss = self.evaluate()
                    val_ppl  = math.exp(min(val_loss, 20))
                    print(f'  >> VAL loss={val_loss:.4f} | ppl={val_ppl:.1f}')
                    wandb.log({
                        'val/loss': val_loss,
                        'val/ppl' : val_ppl,
                        'step'    : self.step,
                    })
                    if val_loss < self.best_val:
                        self.best_val = val_loss
                        self.save(val_loss)

        wandb.finish()
        print(f'\nDone! Best val loss={self.best_val:.4f} | ppl={math.exp(self.best_val):.1f}')


trainer = Trainer(model, train_loader, val_loader, cfg)
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Training 15.0M params for 3 epochs...


/tmp/ipykernel_7029/529182936.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
/tmp/ipykernel_7029/529182936.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == 'cuda')):


Ep 1 | Step    50 | loss=1.9201 | ppl=6.8 | lr=6.28e-05 | 62.4k tok/s
Ep 1 | Step   100 | loss=1.8187 | ppl=6.2 | lr=1.27e-04 | 60.4k tok/s
Ep 1 | Step   150 | loss=1.8684 | ppl=6.5 | lr=1.91e-04 | 62.0k tok/s
Ep 1 | Step   200 | loss=1.9827 | ppl=7.3 | lr=2.55e-04 | 62.3k tok/s


/tmp/ipykernel_7029/529182936.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == 'cuda')):


  >> VAL loss=1.9177 | ppl=6.8
  Checkpoint saved — val_loss=1.9177
Ep 1 | Step   250 | loss=2.0239 | ppl=7.6 | lr=3.00e-04 | 45.8k tok/s
Ep 1 | Step   300 | loss=1.9629 | ppl=7.1 | lr=3.00e-04 | 64.6k tok/s
Ep 1 | Step   350 | loss=1.9216 | ppl=6.8 | lr=3.00e-04 | 65.1k tok/s
Ep 1 | Step   400 | loss=1.8835 | ppl=6.6 | lr=2.99e-04 | 65.1k tok/s
  >> VAL loss=1.9385 | ppl=6.9
Ep 1 | Step   450 | loss=2.0085 | ppl=7.5 | lr=2.98e-04 | 47.5k tok/s
Ep 1 | Step   500 | loss=1.9515 | ppl=7.0 | lr=2.98e-04 | 65.1k tok/s
Ep 1 | Step   550 | loss=1.9007 | ppl=6.7 | lr=2.97e-04 | 64.3k tok/s
Ep 1 | Step   600 | loss=1.9518 | ppl=7.0 | lr=2.96e-04 | 64.3k tok/s
  >> VAL loss=1.9300 | ppl=6.9
Ep 1 | Step   650 | loss=1.8587 | ppl=6.4 | lr=2.94e-04 | 46.7k tok/s
Ep 1 | Step   700 | loss=1.9762 | ppl=7.2 | lr=2.93e-04 | 63.9k tok/s
Ep 1 | Step   750 | loss=1.9972 | ppl=7.4 | lr=2.91e-04 | 64.0k tok/s
Ep 1 | Step   800 | loss=2.0253 | ppl=7.6 | lr=2.89e-04 | 64.3k tok/s
  >> VAL loss=1.9170 | ppl=6.8

step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇██
train/loss,▅▇▇▆▆▅▇▇▅▄▅▅▄█▇▅▅▄▄▅▄▄▃▃▃▄▄▄▁▃▂▂▄▂▁▁▁▂▂▂
train/lr,▂▄▇██████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
train/ppl,▄▅▇▆▇▇▇▄▅▄█▆▆▅▃▄▃▂▂▃▄▃▄▄▃▃▄▄▁▂▂▄▃▂▂▁▁▂▂▂
train/tok_per_s,▇███████▁███▁████▆▁██▁██▁█▁▇▁▁▁█████▁▁██
val/loss,▇██▇▇▆▆▅▅▄▄▄▃▃▂▂▂▂▂▁▁▁▁
val/ppl,▇██▇▇▆▆▅▅▄▄▄▃▃▂▂▂▂▂▁▁▁▁
step,4650
train/loss,1.69261
train/lr,3e-05
train/ppl,5.43366



Done! Best val loss=1.7401 | ppl=5.7


### Gradio Demo (public link)

In [19]:
!pip install gradio -q

import gradio as gr

def gradio_generate(prompt, max_tokens, temperature, top_k):
    if not prompt.strip():
        return 'Please enter a prompt.'
    try:
        return generate_code(
            prompt,
            max_tokens=int(max_tokens),
            temperature=float(temperature),
            top_k=int(top_k)
        )
    except Exception as e:
        return f'Error: {e}'

demo = gr.Interface(
    fn=gradio_generate,
    inputs=[
        gr.Textbox(
            lines=4,
            placeholder='def binary_search(arr, target):',
            label='Python prompt'
        ),
        gr.Slider(50,  400, value=200, step=10,  label='Max tokens'),
        gr.Slider(0.1, 1.5, value=0.8, step=0.1, label='Temperature'),
        gr.Slider(10,  100, value=50,  step=5,   label='Top-k'),
    ],
    outputs=gr.Textbox(lines=20, label='Generated Python code'),
    title='MiniLLM — Python Code Generator',
    description='A 15M parameter GPT-style transformer trained on 400k Python functions from GitHub. Tokenizer, attention, and training loop all built from scratch.',
    examples=[
        ['def binary_search(arr, target):', 200, 0.8, 50],
        ['class LinkedList:\n    def __init__(self):', 200, 0.7, 40],
        ['def quicksort(arr):\n    """Sort using quicksort."""', 200, 0.9, 60],
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4721f2f906a49e70dd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
